# 미사용 TS01 문장 정렬 — Colab GPU (TODOS 3b)

**왜.** 12차·A-1이 "낯선 앱 단어의 남은 이득은 그 단어의 예시에서 나온다"고 했다. 아직 정렬하지
않은 TS01 문장 녹음에 앱 표적 단어가 **112종 · 5869번** 들어 있다(대장 4절 정정). 이 노트북은
그 문장들을 문장 세그먼트로 자른다. 그다음 로컬에서 `cut_word_segments.py`로 앱 단어 구간을
잘라 단독 단어 예시를 만든다(13차 재료).

**무엇을 하나** — VS01 정렬 노트북과 같은 파이프라인이다.
1. `align_608.py --wordlist auto` — whisper `small` 단어 타임스탬프로 문장 세그먼트를 자른다
2. `refine_segments.py --narrative refine` — 에너지로 경계를 다시 잡는다 (**필수**)
3. 결과를 드라이브로 회수 → 로컬에서 병합·패키징

**범위와 순서.** train 화자와 신규 화자의 파일 **209개 · 178시간**만 있다. dev·test 화자의 파일
69개는 뺐다 — 학습에 넣으면 누수, 평가에 넣으면 시험지가 바뀐다(`build_ts01_rest_upload.py`).
매니페스트는 **앱 단어가 녹음 1시간당 많이 나오는 순서**다. 앞 52파일(40시간)에 앱 단어의
47%, 앞 104파일(92시간)에 79%가 있다. 세션이 끊겨도 가장 쓸모 있는 파일부터 끝나 있다.

**준비물(드라이브 `MyDrive/`).**
- `ts01rest_part01.zip` … — 16kHz 모노 FLAC 조각(각 약 1GB). **앞 조각부터** 올린다. 다 안 올려도
  된다 — 이 노트북은 올라온 조각의 파일만 정렬한다
- `align_manifest_ts01rest.jsonl` — 우선순위 순 매니페스트
- `ts01rest_scripts.zip` — 스크립트(커밋 기록은 안의 `VERSION.txt`)

> ⚠️ **정렬 결과는 데이터가 된다.** 이전 배치와 같은 `openai-whisper==20250625` · `small`로 핀을 박는다.

## 셀 1 — GPU 확인 (T4가 떠야 한다)

In [ ]:
!nvidia-smi

## 셀 2 — 설치

`openai-whisper`만 있으면 된다(ffmpeg·numpy는 Colab에 이미 있다). **버전을 핀으로 박는다.**

In [ ]:
WHISPER_PIN = "20250625"   # 이전 배치와 동일. 바꾸면 경계가 달라진다

!pip install -q "openai-whisper=={WHISPER_PIN}"

import importlib.metadata as _m, torch, subprocess
got = _m.version("openai-whisper")
print("whisper:", got, "| torch:", torch.__version__, "| cuda:", torch.cuda.is_available())
print("ffmpeg :", subprocess.run(["ffmpeg","-version"], capture_output=True, text=True)
      .stdout.splitlines()[0])
assert got == WHISPER_PIN, f"핀이 안 먹었다: {got}"
assert torch.cuda.is_available(), "GPU가 없다 — 런타임 유형을 T4로 바꿔라"

## 셀 3 — 드라이브 마운트 + 오디오 풀기 (FLAC → WAV, 파일별로)

오디오는 `/content`(빠름)에 풀고 **산출물은 드라이브**에 쓴다.

- **파일별로 하나씩** 꺼낸다. VS01 때 `extractall`로 한 번에 크게 읽다가 드라이브 FUSE가
  끊겼다(`Errno 107`). 이미 있는 파일은 건너뛰므로 다시 실행해도 빠르다.
- **FLAC을 WAV로 되돌린다.** `refine_segments.py`가 `wave`로 부모 오디오를 읽기 때문이다.
  매니페스트의 `expected_audio_relpath`가 그 WAV 경로다.
- **올라온 조각의 파일만** 푼다. 세션이 새 VM이면 `/content`가 비는데, refine은 이미 정렬한
  파일의 부모 오디오도 필요하므로 **올라온 전부**를 채운다(VS01에서 범위를 좁혔다가 refine이 0개를 냈다).

In [ ]:
import os, zipfile, json, time, glob, subprocess
from google.colab import drive
drive.mount("/content/drive")

D     = "/content/drive/MyDrive"
MAN   = f"{D}/align_manifest_ts01rest.jsonl"
SZIP  = f"{D}/ts01rest_scripts.zip"
ROOT  = "/content/608-root"                 # audio-root. 매니페스트 경로 ts01rest/<id>.wav가 이 아래
OUT   = f"{D}/ts01rest_segs"                # 산출물(드라이브 = 재개 가능)
os.makedirs(f"{ROOT}/ts01rest", exist_ok=True); os.makedirs(OUT, exist_ok=True)


def _retry(fn, what, tries=3):
    """드라이브 FUSE는 간헐적으로 끊긴다(Errno 107). 몇 번 다시 해본다."""
    for k in range(tries):
        try:
            return fn()
        except OSError as e:
            print(f"  [{k+1}/{tries}] {what} 실패: {e}")
            time.sleep(5)
    raise SystemExit(f"{what}에 계속 실패한다. [런타임 → 세션 다시 시작] 후 이 셀부터 다시 실행해라.")


_retry(lambda: zipfile.ZipFile(SZIP).extractall(ROOT), "스크립트 풀기")
print(open(f"{ROOT}/VERSION.txt", encoding="utf-8").read().strip())
rows = [json.loads(l) for l in open(MAN, encoding="utf-8")]
parts = sorted(glob.glob(f"{D}/ts01rest_part*.zip"))
print(f"매니페스트 {len(rows)}행 · 드라이브의 조각 {len(parts)}개")
assert parts, "ts01rest_part01.zip부터 올려라"

TMP = "/content/_flac"; os.makedirs(TMP, exist_ok=True)
n_new = 0
for p in parts:
    with zipfile.ZipFile(p) as z:
        for m in z.namelist():                  # ts01rest/<stem>.flac
            wav = os.path.join(ROOT, m[:-5] + ".wav")
            if os.path.exists(wav):
                continue
            _retry(lambda m=m: z.extract(m, TMP), m)
            src = os.path.join(TMP, m)
            subprocess.run(["ffmpeg", "-v", "error", "-y", "-i", src, "-ac", "1", "-ar", "16000",
                            "-sample_fmt", "s16", wav + ".part.wav"], check=True)
            os.replace(wav + ".part.wav", wav)  # 반쯤 쓴 WAV가 "있음"으로 잡히지 않게
            os.remove(src)
            n_new += 1
            if n_new % 20 == 0:
                print(f"  새로 푼 파일 {n_new}", flush=True)
    print(f"  {os.path.basename(p)} 확인")

present = [r for r in rows if os.path.exists(os.path.join(ROOT, r["expected_audio_relpath"]))]
N_PRESENT = len(present)
h = sum(r["play_time_sec"] for r in present) / 3600
print(f"{os.linesep}디스크에 있는 오디오 {N_PRESENT}/{len(rows)}개 · {h:.0f}시간 (이번에 새로 {n_new})")
print("→ 이 파일들만 정렬한다. 나머지 조각은 나중에 올리고 셀 3부터 다시 돌리면 이어진다.")

## 셀 3.5 — 검문: 전부 문장 모드로 도는가 (**건너뛰지 마라**)

`align_608`은 단어/문장 판별을 `asr_eval/baseline_asr.py`의 `task_type`으로 한다. 임포트가 실패해도
죽지 않고 전부 문장 모드로 돈다(VS01에서 70분을 버린 원인). 이번엔 **전부 문장**이 맞지만,
임포트 실패로 우연히 맞는 것과 판별이 제대로 돌아 맞는 것은 다르다 — 여기서 확인한다.

In [ ]:
import sys, os, importlib
sys.path.insert(0, os.path.join(ROOT, "asr_eval"))
for m in ("baseline_asr", "metrics", "split_speakers"):
    if m in sys.modules: importlib.reload(sys.modules[m])
try:
    from baseline_asr import task_type
except Exception as e:
    raise SystemExit(f"task_type 임포트 실패: {type(e).__name__}: {e} — ts01rest_scripts.zip을 확인해라.")

kinds = [task_type(r.get("transcript") or "") for r in rows]
print(f"task_type 판정: 문장 {kinds.count('narrative')} · 단어 {kinds.count('wordlist')} / {len(rows)}")
assert kinds.count("narrative") == len(rows), "전부 narrative여야 한다 — 매니페스트가 문장 파일만 담고 있다"

sys.path.insert(0, ROOT)
from align_608 import _sentences
n_sent = [len(_sentences(r.get("transcript") or "")) for r in rows]
print(f"파일당 문장 수: 최소 {min(n_sent)} · 중앙 {sorted(n_sent)[len(n_sent)//2]} · 최대 {max(n_sent)}")
print("→ 문장 모드가 맞다. 정렬로 넘어가도 좋다.")

## 셀 3.6 — 산출물 치우기 (평소엔 **아무것도 안 한다**)

처음부터 다시 정렬할 때만 `RESET_SEGS = True`. `align_608`은 `_attempted.txt`로 재개하므로 이 셀이
기본으로 지우면 세션이 끊겨 다시 돌릴 때마다 진행분이 날아간다. 지우지 않고 옆으로 치운다.

In [ ]:
RESET_SEGS = False   # 처음부터 다시 정렬할 때만 True로

import os, shutil, time, json
if not RESET_SEGS:
    att = f"{OUT}/_attempted.txt"
    n_att = sum(1 for l in open(att, encoding="utf-8") if l.strip()) if os.path.exists(att) else 0
    n_seg = sum(1 for _ in open(f"{OUT}/segments.jsonl", encoding="utf-8")) if os.path.exists(f"{OUT}/segments.jsonl") else 0
    print(f"유지: 시도한 파일 {n_att} · 세그먼트 {n_seg}개 (다시 정렬하려면 RESET_SEGS = True)")
else:
    if os.path.exists(OUT):
        aside = f"{OUT}-폐기-{time.strftime('%m%d-%H%M')}"
        shutil.move(OUT, aside)
        print(f"치웠다: {aside}{os.linesep}되돌리려면: !mv '{aside}' '{OUT}'")
    os.makedirs(OUT, exist_ok=True)

## 셀 4 — 정렬 (오래 걸린다 · 끊기면 셀 1~3을 다시 돌리고 이 셀만 다시)

- `--model small` · `--wordlist auto` : 이전 배치와 동일. 바꾸지 마라
- 매니페스트 순서(앱 단어 밀도순)대로 돈다. 디스크에 있는 파일만 한다
- 예상: VS01은 41시간 오디오였다. 이번엔 올린 만큼(최대 178시간)이라 **여러 세션**이 걸린다.
  한 세션에서 끝낼 생각 말고, 끊기면 이어서 돌린다

In [ ]:
import time, subprocess, os, json
t0 = time.time()
cmd = ["python", f"{ROOT}/align_608.py",
       "--manifest", MAN,
       "--audio-root", ROOT,
       "--out-dir", OUT,
       "--model", "small",
       "--wordlist", "auto"]
print(" ".join(cmd), flush=True)
subprocess.run(cmd, check=False)      # 실패해도 아래 진행 상황은 보고 싶다
print(f"{os.linesep}경과 {(time.time()-t0)/60:.1f}분")

# **"처리한 파일"은 _attempted.txt로 센다** — 세그먼트 0개로 끝난 파일은 segments.jsonl에 안 남는다
att = f"{OUT}/_attempted.txt"
attempted = {l.strip() for l in open(att, encoding="utf-8") if l.strip()} if os.path.exists(att) else set()
segs = [json.loads(l) for l in open(f"{OUT}/segments.jsonl", encoding="utf-8")] if os.path.exists(f"{OUT}/segments.jsonl") else []
has = {s["parent_file_id"] for s in segs}
dur = sum(s["end"] - s["start"] for s in segs) / 3600
print(f"세그먼트 {len(segs)}개 · {dur:.1f}시간 · 시도한 파일 {len(attempted)}/{N_PRESENT} (세그 0개 {len(attempted - has)})")
# 기존 문장 배치: 문장 1873개 → 세그먼트 1569개(전사의 82%). 파일당 수 개~수십 개가 정상이다.
# 파일당 1개에 가깝다면 문장 분할이 안 된 것이다 — 멈추고 원인을 본다.
if has and len(segs) / len(has) < 3:
    raise SystemExit(f"파일당 세그먼트 {len(segs)/len(has):.1f}개 — 문장 분할이 안 됐다. 셀 3.5를 보라.")
if len(attempted) < N_PRESENT:
    print("→ 아직 남았다. 이 셀을 다시 실행하면 이어서 간다.")
else:
    print("→ 올린 파일 전부 시도 완료. 셀 5(재정렬)로.")

## 셀 5 — 재정렬 (필수)

문장 클립도 앞뒤 침묵이 평균 57%이고 30%는 끝이 잘려 있다(대장 3-2절). `--narrative refine`을
**명시해야** 문장까지 처리한다(기본은 skip). `--resume`으로 이어서 한다.

In [ ]:
import subprocess, json, collections, time, os

_rows = [json.loads(l) for l in open(f"{OUT}/segments.jsonl", encoding="utf-8")]
_parents = {r["parent_file_id"] for r in _rows}
_man = {json.loads(l)["file_id"]: json.loads(l)["expected_audio_relpath"] for l in open(MAN, encoding="utf-8")}
_missing = [p for p in _parents if not os.path.exists(os.path.join(ROOT, _man.get(p, p)))]
if _missing:
    raise SystemExit(f"부모 {len(_missing)}/{len(_parents)}개의 오디오가 없다(예: {_missing[:2]}). 셀 3을 다시 실행해라.")
print(f"오디오 점검 통과: 부모 {len(_parents)}개 전부 있음")

REF = OUT + "_refined"
t0 = time.time()
subprocess.run(["python", f"{ROOT}/asr_eval/refine_segments.py",
                "--segments", f"{OUT}/segments.jsonl",
                "--audio-root", ROOT,
                "--out-dir", REF,
                "--narrative", "refine",
                "--resume"], check=False)
print(f"경과 {(time.time()-t0)/60:.1f}분")

rows_r = [json.loads(l) for l in open(f"{REF}/segments.jsonl", encoding="utf-8")]
c = collections.Counter(r.get("task_type") for r in rows_r)
dur = sum(r["end"] - r["start"] for r in rows_r)
print(f"재정렬 후 {len(rows_r)}개 · {dict(c)} · 총 {dur/3600:.1f}시간")
print("전부 narrative여야 정상이다. wordlist가 섞였으면 --wordlist-file-frac 판정을 확인해라.")

## 셀 6 — 회수 (문장 세그먼트 wav를 로컬에서 zip → 드라이브에 파일 하나로)

In [ ]:
import shutil, os, time

# 드라이브의 파일 수만 개를 직접 zip하면 FUSE가 끊긴다(VS01 셀 6, Errno 107).
# ① /content로 파일별 재시도 복사 ② 로컬에서 zip ③ zip 하나만 드라이브에 올린다.
LOCAL_REF = "/content/ts01rest_refined_local"
os.makedirs(LOCAL_REF, exist_ok=True)


def _retry(fn, what, tries=5, wait=5):
    for k in range(tries):
        try:
            return fn()
        except OSError as e:
            print(f"  [{k+1}/{tries}] {what} 실패: {e}")
            time.sleep(wait)
    raise SystemExit(f"{what}에 계속 실패한다. 드라이브를 다시 마운트하고 이 셀부터 재실행해라.")


all_files = [(os.path.join(dp, f), os.path.relpath(os.path.join(dp, f), REF))
             for dp, _, fs in os.walk(REF) for f in fs]
print(f"드라이브의 파일 {len(all_files)}개 → {LOCAL_REF}")
for n, (full, rel) in enumerate(all_files, 1):
    dst = os.path.join(LOCAL_REF, rel)
    os.makedirs(os.path.dirname(dst), exist_ok=True)
    if not (os.path.exists(dst) and os.path.getsize(dst) == os.path.getsize(full)):
        _retry(lambda full=full, dst=dst: shutil.copy2(full, dst), rel)
    if n % 1000 == 0 or n == len(all_files):
        print(f"  {n}/{len(all_files)}", flush=True)

local_zip = shutil.make_archive("/content/ts01rest_segs_refined", "zip", LOCAL_REF)
drive_zip = f"{D}/ts01rest_segs_refined.zip"
_retry(lambda: shutil.copy2(local_zip, drive_zip), "드라이브 업로드")
print(f"회수용 zip: {drive_zip}  {os.path.getsize(drive_zip)/2**20:.0f} MB")
print()
print("로컬에서 할 일:")
print("  1. ts01rest_segs_refined.zip을 받아 ~/Downloads/608-audio-work/_segs_ts01rest_refined/ 로 풀기")
print("  2. cut_word_segments.py로 앱 단어 구간 잘라내기 (정렬 품질 검수 결과를 본 뒤)")
print("  3. 13차 설계·판정 규칙을 대장에 등록한 뒤 패키징")